# <center> Differential expression analysis within groups

## <center> Description

- env: squidpy_v06
- Subset data and perform DEA for:
    - HR case, IFNG+ vs INFG-
    - HR control, IFNG+ vs INFG-

## <center> Set-up

#### **libs**

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import decoupler as dc
import sys
from pathlib import Path
from scipy import sparse
## for vst
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

# When this notebook is exported and run as a script, add the project root to sys.path
# so local workflow.* imports behave like they do in the notebook kernel.
if "__file__" in globals():
    PROJECT_ROOT = Path(__file__).resolve().parents[3]
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

## helper functions
from workflow.scripts.funs.helper_funs import get_project_root

## plotting
# import marsilea as ma
# import marsilea.plotter as mp
# from matplotlib.colors import NoNorm # for marsilea dotplot
import plotnine as p9
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

#### **paths**

In [2]:
from pathlib import Path

if "snakemake" in globals():
    # in_metadata = Path(snakemake.input["in_metadata"])
    # out_metadata = Path(snakemake.output["out_metadata"])
    print('Hello World')
else:
    ## get paths
    ROOT = get_project_root(globals())
    
    ## params
    idx=1 # 0-1
    runName_lst=['HR_case__IFNGpos_vs_IFNGneg', 'HR_control__IFNGpos_vs_IFNGneg']
    run_name=runName_lst[idx]
    params_dict = {
        ### run_name: {params}
        'HR_case__IFNGpos_vs_IFNGneg': {
            'filter_query':"disease_group=='HR_case'", 
            'design_formula':'~ individualID + condition_group', # this design is similar to a paired t-test, since samples are repeated by patients
            'contrast':['condition_group', 'IFNG_pos', 'IFNG_neg'] },
        'HR_control__IFNGpos_vs_IFNGneg':{
            'filter_query':"disease_group=='HR_control'", 
            'design_formula':'~ individualID + condition_group', 
            'contrast':['condition_group', 'IFNG_pos', 'IFNG_neg']},
    }
    filter_query=params_dict[run_name]['filter_query']
    design_formula=params_dict[run_name]['design_formula']
    contrast=params_dict[run_name]['contrast']
    
    ## input
    in_adata = ROOT / "output/preprocessing/objects/QC/adata.h5ad"
    
    ## output
    ### objects
    out_deg = ROOT / f"output/DEA/objects/pyDESeq2_byGrp/{run_name}/deg.tsv"
    out_deg.parent.mkdir(parents=True, exist_ok=True) # make sure the output folder exists
    ### plots
    out_plots = ROOT / f"output/DEA/plots/pyDESeq2_byGrp/{run_name}"
    out_plots.mkdir(parents=True, exist_ok=True)


#### **reproducibility**

In [3]:
import random
import numpy as np
import torch
import scanpy.external as sce

SEED = 0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## <center> Main

#### **Load data**

In [4]:
adata = sc.read_h5ad(in_adata)

adata

AnnData object with n_obs × n_vars = 66 × 19557
    obs: 'disease_group', 'QC_group', 'condition_group', 'grp', 'leiden_0.05', 'leiden_0.1', 'leiden_0.15', 'leiden_0.2'
    var: 'genesymbol', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'QC_group_colors', 'condition_group_colors', 'disease_group_colors', 'grp_colors', 'hvg', 'leiden_0.05', 'leiden_0.1', 'leiden_0.15', 'leiden_0.15_colors', 'leiden_0.2', 'log1p', 'neighbors', 'pca', 'rank_obsm', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'cts.deseq2_norm', 'cts.deseq2_norm_log', 'cts.raw', 'cts.vst'
    obsp: 'connectivities', 'distances'

In [5]:
# remove group B

adata = adata[adata.obs['QC_group']=='A'].copy()

adata

AnnData object with n_obs × n_vars = 50 × 19557
    obs: 'disease_group', 'QC_group', 'condition_group', 'grp', 'leiden_0.05', 'leiden_0.1', 'leiden_0.15', 'leiden_0.2'
    var: 'genesymbol', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'QC_group_colors', 'condition_group_colors', 'disease_group_colors', 'grp_colors', 'hvg', 'leiden_0.05', 'leiden_0.1', 'leiden_0.15', 'leiden_0.15_colors', 'leiden_0.2', 'log1p', 'neighbors', 'pca', 'rank_obsm', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'cts.deseq2_norm', 'cts.deseq2_norm_log', 'cts.raw', 'cts.vst'
    obsp: 'connectivities', 'distances'

In [6]:
# filter using rule

adata = adata[adata.obs.query(filter_query).index].copy()

adata

AnnData object with n_obs × n_vars = 26 × 19557
    obs: 'disease_group', 'QC_group', 'condition_group', 'grp', 'leiden_0.05', 'leiden_0.1', 'leiden_0.15', 'leiden_0.2'
    var: 'genesymbol', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'QC_group_colors', 'condition_group_colors', 'disease_group_colors', 'grp_colors', 'hvg', 'leiden_0.05', 'leiden_0.1', 'leiden_0.15', 'leiden_0.15_colors', 'leiden_0.2', 'log1p', 'neighbors', 'pca', 'rank_obsm', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'cts.deseq2_norm', 'cts.deseq2_norm_log', 'cts.raw', 'cts.vst'
    obsp: 'connectivities', 'distances'

In [7]:
# add individualID

col_dict = {
    'SD1': 'individual_01','SD2': 'individual_01','SD3': 'individual_02','SD4': 'individual_02',
    'SD5': 'individual_03','SD6': 'individual_03','SD7': 'individual_04','SD8': 'individual_04',
    'SD9': 'individual_05','SD10': 'individual_05','SD11': 'individual_06','SD12': 'individual_06',
    'SD13': 'individual_07','SD14': 'individual_07','SD15': 'individual_08','SD16': 'individual_08',
    'SD17': 'individual_09','SD18': 'individual_09','SD19': 'individual_10','SD20': 'individual_10',
    'SD21': 'individual_11','SD22': 'individual_11','SD23': 'individual_12','SD24': 'individual_12',
    'SD25': 'individual_13','SD26': 'individual_13','SD27': 'individual_14','SD28': 'individual_14',
    'SD29': 'individual_15','SD30': 'individual_15','SD31': 'individual_16','SD32': 'individual_16',
    'SD33': 'individual_17','SD34': 'individual_17','SD35': 'individual_18','SD36': 'individual_18',
    'SD37': 'individual_19','SD38': 'individual_19','SD39': 'individual_20','SD40': 'individual_20',
    'SD41': 'individual_21','SD42': 'individual_21','SD43': 'individual_22','SD44': 'individual_22',
    'SD45': 'individual_23','SD46': 'individual_23','SD47': 'individual_24','SD48': 'individual_24',
    'SD49': 'individual_25','SD50': 'individual_25','SD51': 'individual_26','SD52': 'individual_26',
    'SD53': 'individual_27','SD54': 'individual_27','SD55': 'individual_28','SD56': 'individual_28',
    'SD57': 'individual_29','SD58': 'individual_29','SD59': 'individual_30','SD60': 'individual_30',
    'SD61': 'individual_31','SD62': 'individual_31','SD63': 'individual_32','SD64': 'individual_32',
    'SD65': 'individual_33','SD66': 'individual_33',
}
adata.obs['individualID']=adata.obs.index.map(col_dict)
adata.obs['individualID']=adata.obs['individualID'].astype('category')

adata.obs.head()

,disease_group,QC_group,condition_group,grp,leiden_0.05,leiden_0.1,leiden_0.15,leiden_0.2,individualID
sample_id,,,,,,,,,
SD41,HR_control,A,IFNG_neg,HR_control__IFNG_neg,0,0,0,2,individual_21
SD43,HR_control,A,IFNG_neg,HR_control__IFNG_neg,0,0,0,0,individual_22
SD45,HR_control,A,IFNG_neg,HR_control__IFNG_neg,0,0,0,2,individual_23
SD47,HR_control,A,IFNG_neg,HR_control__IFNG_neg,0,0,0,0,individual_24
SD49,HR_control,A,IFNG_neg,HR_control__IFNG_neg,0,0,0,0,individual_25


In [8]:
# make sure counts are raw

adata.X = adata.layers['cts.raw'].copy()

In [9]:
# check dist of samples across groups

adata.obs[['disease_group', 'condition_group']].value_counts()

disease_group  condition_group
HR_control     IFNG_neg           13
               IFNG_pos           13
Name: count, dtype: int64

In [10]:
# # create the combined DEA group used for the simpler one-factor design

# # These are the only levels used in this 2 x 2 test analysis.
# condition_levels = ["IFNG_neg", "IFNG_pos"]
# disease_levels = ["HR_control", "HR_case"]

# # Keep only samples belonging to the requested 2 x 2 design.
# adata = adata[
#     adata.obs["condition_group"].isin(condition_levels)
#     & adata.obs["disease_group"].isin(disease_levels)
# ].copy()

# # Store the original factors as categorical for clear ordering in summaries.
# adata.obs["condition_group"] = pd.Categorical(
#     adata.obs["condition_group"],
#     categories=condition_levels,
# )
# adata.obs["disease_group"] = pd.Categorical(
#     adata.obs["disease_group"],
#     categories=disease_levels,
# )

# # Merge disease_group and condition_group into the requested combined factor.
# # The separator is double underscore and the disease group comes first, e.g. HR_case__IFNG_pos.
# DEA_group_levels = [
#     "HR_control__IFNG_pos",
#     "HR_case__IFNG_pos",
#     "HR_control__IFNG_neg",
#     "HR_case__IFNG_neg",
# ]
# adata.obs["DEA_group"] = pd.Categorical(
#     adata.obs["disease_group"].astype(str) + "__" + adata.obs["condition_group"].astype(str),
#     categories=DEA_group_levels,
# )

# # Sanity checks before model fitting.
# assert adata.n_obs > 0, "No samples left after subsetting to the requested DEA groups."
# assert not adata.obs[["condition_group", "disease_group", "DEA_group"]].isna().any().any(), "Unexpected factor levels found."

# dea_sample_counts = (
#     adata.obs[["DEA_group", "disease_group", "condition_group"]]
#     .value_counts(sort=False)
#     .rename("n_samples")
#     .reset_index()
# )

# # A one-factor model with all combined levels needs samples in every level kept here.
# assert (dea_sample_counts["n_samples"] > 0).all(), "The combined DEA_group design has an empty group."

# dea_sample_counts


## <center> DEA

In [11]:
# prepare data for pyDESeq2

# Use the raw count layer prepared in QC.ipynb.
# Genes were already filtered there, so no additional expression filtering is applied here.
adata_dds = adata.copy()
adata_dds.X = adata_dds.layers["cts.raw"].copy()

# pyDESeq2 expects an in-memory dense integer count matrix.
# The rounding is a safety step in case the count layer was stored as floating point.
if sparse.issparse(adata_dds.X):
    adata_dds.X = adata_dds.X.toarray()
adata_dds.X = np.rint(np.asarray(adata_dds.X)).astype(int)

# Keep all QC-filtered genes, but remove any accidental all-zero genes because DESeq2 cannot estimate them.
nonzero_genes = np.asarray(adata_dds.X.sum(axis=0)).ravel() > 0
print(f"Genes before all-zero safety check: {adata_dds.n_vars}")
print(f"Genes after all-zero safety check: {nonzero_genes.sum()}")
adata_dds = adata_dds[:, nonzero_genes].copy()

adata_dds


Genes before all-zero safety check: 19557
Genes after all-zero safety check: 19555


AnnData object with n_obs × n_vars = 26 × 19555
    obs: 'disease_group', 'QC_group', 'condition_group', 'grp', 'leiden_0.05', 'leiden_0.1', 'leiden_0.15', 'leiden_0.2', 'individualID'
    var: 'genesymbol', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'QC_group_colors', 'condition_group_colors', 'disease_group_colors', 'grp_colors', 'hvg', 'leiden_0.05', 'leiden_0.1', 'leiden_0.15', 'leiden_0.15_colors', 'leiden_0.2', 'log1p', 'neighbors', 'pca', 'rank_obsm', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'cts.deseq2_norm', 'cts.deseq2_norm_log', 'cts.raw', 'cts.vst'
    obsp: 'connectivities', 'distances'

In [12]:
# fit the DESeq2 model 

dds = DeseqDataSet(
    adata=adata_dds,
    design=design_formula,
    refit_cooks=True,
    n_cpus=1,  # kept at 1 because this environment has shown pyDESeq2 multiprocessing issues
)

# Run the standard DESeq2 workflow: size factors, dispersions, LFCs, and Cook's refitting if needed.
dds.deseq2()

# Check the design matrix columns so the contrast below is easy to audit.
dds.obsm["design_matrix"].head()


Using None as control genes, passed at DeseqDataSet initialization


Fitting size factors...
... done in 0.03 seconds.

Fitting dispersions...
... done in 17.57 seconds.

Fitting dispersion trend curve...
... done in 0.38 seconds.

Fitting MAP dispersions...
... done in 19.88 seconds.

Fitting LFCs...
... done in 11.77 seconds.

Calculating cook's distance...
... done in 0.03 seconds.

Replacing 0 outlier genes.



,Intercept,individualID[T.individual_22],individualID[T.individual_23],individualID[T.individual_24],individualID[T.individual_25],individualID[T.individual_26],individualID[T.individual_27],individualID[T.individual_28],individualID[T.individual_29],individualID[T.individual_30],individualID[T.individual_31],individualID[T.individual_32],individualID[T.individual_33],condition_group[T.IFNG_pos]
sample_id,,,,,,,,,,,,,,
SD41,1.0,0,0,0,0,0,0,0,0,0,0,0,0,0
SD43,1.0,1,0,0,0,0,0,0,0,0,0,0,0,0
SD45,1.0,0,1,0,0,0,0,0,0,0,0,0,0,0
SD47,1.0,0,0,1,0,0,0,0,0,0,0,0,0,0
SD49,1.0,0,0,0,1,0,0,0,0,0,0,0,0,0


In [13]:
# extract the requested DEA_group contrast

# This compares HR_case__IFNG_pos against HR_control__IFNG_pos.
# Therefore, a positive log2FC means higher expression in HR_case__IFNG_pos.
stat_res = DeseqStats(
    dds,
    contrast=contrast,
    alpha=0.05,
    cooks_filter=True,
    independent_filter=True,
    n_cpus=1,
)
stat_res.summary()

# Format results and add gene symbols when they are available in adata.var.
deg = stat_res.results_df.copy()
deg.index.name = "gene_id"

if "genesymbol" in adata_dds.var.columns:
    deg.insert(0, "genesymbol", adata_dds.var["genesymbol"].reindex(deg.index).astype(str))

deg["contrast"] = run_name
deg["significant_pvalue_0.05"] = deg["pvalue"] < 0.05
deg["significant_padj_0.05"] = deg["padj"] < 0.05

# Keep the most significant genes at the top of the saved table.
deg = deg.sort_values(["padj", "pvalue"], na_position="last")

deg.head(20)


Running Wald tests...


Log2 fold change & Wald test p-value: condition_group IFNG_pos vs IFNG_neg
                    baseMean  log2FoldChange     lfcSE      stat  \
gene_id                                                            
ENSG00000000003  1747.795759       -0.255615  0.048126 -5.311359   
ENSG00000000005     1.099562       -0.734969  1.520093 -0.483502   
ENSG00000000419   985.482801       -0.021653  0.041669 -0.519653   
ENSG00000000457   177.674873        0.415815  0.098701  4.212890   
ENSG00000000460    76.663876        0.098952  0.107241  0.922706   
...                      ...             ...       ...       ...   
ENSG00000289612     3.013582       -1.303740  0.683029 -1.908763   
ENSG00000289613     6.166238       -0.178169  0.334779 -0.532198   
ENSG00000289614    19.153164        0.133538  0.211542  0.631262   
ENSG00000289621     4.209737       -0.427781  0.477612 -0.895668   
ENSG00000289624     7.506854        0.658343  0.376393  1.749086   

                       pvalue      padj 

... done in 2.56 seconds.



,genesymbol,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,contrast,significant_pvalue_0.05,significant_padj_0.05
gene_id,,,,,,,,,,
ENSG00000138755,CXCL9,3374.945550,8.174728,0.253833,32.205146,1.495163e-227,2.753642e-223,HR_control__IFNGpos_vs_IFNGneg,True,True
ENSG00000169245,CXCL10,2778.881911,7.330783,0.252336,29.051647,1.466821e-185,1.350723e-181,HR_control__IFNGpos_vs_IFNGneg,True,True
ENSG00000128335,APOL2,1983.748117,2.830091,0.104085,27.190112,8.501915e-163,5.219326e-159,HR_control__IFNGpos_vs_IFNGneg,True,True
ENSG00000131203,IDO1,2166.026492,8.284604,0.339479,24.403869,1.555856e-131,7.163551e-128,HR_control__IFNGpos_vs_IFNGneg,True,True
ENSG00000120217,CD274,512.160594,5.828198,0.276733,21.060709,1.824237e-98,6.719393e-95,HR_control__IFNGpos_vs_IFNGneg,True,True
ENSG00000169248,CXCL11,638.168153,7.323194,0.347993,21.044073,2.591324e-98,7.954070e-95,HR_control__IFNGpos_vs_IFNGneg,True,True
ENSG00000182326,C1S,1647.438060,2.526253,0.129526,19.503821,1.018790e-84,2.486063e-81,HR_control__IFNGpos_vs_IFNGneg,True,True
ENSG00000159403,C1R,727.001688,2.936909,0.150604,19.500842,1.079899e-84,2.486063e-81,HR_control__IFNGpos_vs_IFNGneg,True,True
ENSG00000177409,SAMD9L,830.000417,5.385325,0.304908,17.662138,8.207107e-70,1.679448e-66,HR_control__IFNGpos_vs_IFNGneg,True,True


In [14]:
# QC plot: P value histogram

# A broadly well-calibrated test often has a roughly uniform null component plus
# enrichment near zero when there is signal. Strong deviations can indicate model or batch issues.
plot_tab = deg.dropna(subset=["pvalue"]).copy()

w, h = 5, 4
p = (
    p9.ggplot(plot_tab, p9.aes(x="pvalue"))
    + p9.geom_histogram(
        breaks=np.arange(0, 1.05, 0.05),
        fill="#4C78A8",
        color="white",
        boundary=0,
    )
    + p9.scale_x_continuous(breaks=np.arange(0, 1.05, 0.1), limits=(0, 1))
    + p9.theme_classic()
    + p9.labs(
        title="P value distribution",
        x="Raw P value",
        y="Number of genes",
    )
    + p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=13),
        axis_title=p9.element_text(size=12),
        axis_text=p9.element_text(size=10),
        figure_size=(w, h),
    )
)

p.save(out_plots / "pvalue_histogram.png", width=w, height=h, limitsize=False)
p.save(out_plots / "pvalue_histogram.pdf", width=w, height=h, limitsize=False)

plt.close()


/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 5 x 4 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/pyDESeq2_byGrp/HR_control__IFNGpos_vs_IFNGneg/pvalue_histogram.png


/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 5 x 4 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/pyDESeq2_byGrp/HR_control__IFNGpos_vs_IFNGneg/pvalue_histogram.pdf


In [15]:
# QC plot: volcano plot with decoupler

# decoupler labels the top significant genes using the dataframe index.
# The y-axis uses adjusted P values so the significance threshold is FDR < 0.05.
# Use gene symbols for labels when available; otherwise fall back to gene IDs.
volcano_tab = deg.dropna(subset=["log2FoldChange", "padj"]).copy()
if "genesymbol" in volcano_tab.columns:
    volcano_tab["plot_label"] = volcano_tab["genesymbol"].where(
        volcano_tab["genesymbol"].notna() & (volcano_tab["genesymbol"] != "nan"),
        volcano_tab.index.astype(str),
    )
else:
    volcano_tab["plot_label"] = volcano_tab.index.astype(str)
volcano_tab = volcano_tab.set_index("plot_label", drop=False)

fig = dc.pl.volcano(
    data=volcano_tab,
    x="log2FoldChange",
    y="padj",
    top=20,
    thr_stat=0.5,
    thr_sign=0.05,
    figsize=(7, 6),
    dpi=120,
    return_fig=True,
)
fig.axes[0].set_title(run_name)
fig.savefig(out_plots / "volcano.png", bbox_inches="tight")
fig.savefig(out_plots / "volcano.pdf", bbox_inches="tight")

plt.close(fig)


In [16]:
# QC plot: number of genes passing raw and adjusted P value thresholds

# Count genes passing the two requested significance thresholds.
sig_counts = pd.DataFrame(
    {
        "threshold": ["Raw P < 0.05", "Padj < 0.05"],
        "n_genes": [
            int((deg["pvalue"] < 0.05).sum()),
            int((deg["padj"] < 0.05).sum()),
        ],
    }
)

w, h = 4.5, 4
p = (
    p9.ggplot(sig_counts, p9.aes(x="threshold", y="n_genes", fill="threshold"))
    + p9.geom_col(color="black", width=0.7, show_legend=False)
    + p9.geom_text(p9.aes(label="n_genes"), va="bottom", size=10, format_string="{:.0f}")
    + p9.scale_fill_manual(["#4C78A8", "#F58518"])
    + p9.theme_classic()
    + p9.labs(
        title="Significant genes",
        x="",
        y="Number of genes",
    )
    + p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=13),
        axis_title=p9.element_text(size=12),
        axis_text_x=p9.element_text(size=10, rotation=20, ha="right"),
        axis_text_y=p9.element_text(size=10),
        figure_size=(w, h),
    )
)

p.save(out_plots / "significant_gene_counts.png", width=w, height=h, limitsize=False)
p.save(out_plots / "significant_gene_counts.pdf", width=w, height=h, limitsize=False)

plt.close()

sig_counts


/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 4.5 x 4 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/pyDESeq2_byGrp/HR_control__IFNGpos_vs_IFNGneg/significant_gene_counts.png
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 4.5 x 4 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/pyDESeq2_byGrp/HR_control__IFNGpos_vs_IFNGneg/significant_gene_counts.pdf


,threshold,n_genes
0,Raw P < 0.05,5356
1,Padj < 0.05,3379


In [17]:
# QC plot: MA plot

# The MA plot shows the relationship between average normalized expression and log2FC.
# It is useful for spotting low-count instability or global shifts.
ma_tab = deg.dropna(subset=["baseMean", "log2FoldChange"]).copy()
ma_tab["is_padj_0.05"] = ma_tab["padj"] < 0.05

w, h = 5.5, 4.5
p = (
    p9.ggplot(ma_tab, p9.aes(x="baseMean", y="log2FoldChange"))
    + p9.geom_point(p9.aes(color="is_padj_0.05"), alpha=0.55, size=0.8)
    + p9.scale_x_log10()
    + p9.scale_color_manual({False: "#7F7F7F", True: "#D62728"})
    + p9.geom_hline(yintercept=0, linetype="dashed")
    + p9.theme_classic()
    + p9.labs(
        title="MA plot",
        x="Mean normalized counts",
        y="log2FC",
        color="Padj < 0.05",
    )
    + p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=13),
        axis_title=p9.element_text(size=12),
        axis_text=p9.element_text(size=10),
        legend_title=p9.element_text(size=10),
        legend_text=p9.element_text(size=9),
        figure_size=(w, h),
    )
)

p.save(out_plots / "ma_plot.png", width=w, height=h, limitsize=False)
p.save(out_plots / "ma_plot.pdf", width=w, height=h, limitsize=False)

plt.close()


/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 5.5 x 4.5 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/pyDESeq2_byGrp/HR_control__IFNGpos_vs_IFNGneg/ma_plot.png
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 5.5 x 4.5 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/pyDESeq2_byGrp/HR_control__IFNGpos_vs_IFNGneg/ma_plot.pdf


#### **Save results**

In [18]:
deg.to_csv(out_deg,sep='\t')